# 🐍 파이썬 정규표현식(Regex) 종합 마스터 가이드

기본적인 메타문자와 핵심 함수부터 고급 기능(그룹, 플래그, 실전 응용)까지 모두 담은 통합 실습 노트북입니다.

In [2]:
import re
print("re 모듈 임포트 완료!")

re 모듈 임포트 완료!


----

## 1. 핵심 메타문자 (기본 문법 요약)

* **문자 클래스 (어떤 글자 하나)**
  * `.` : 줄바꿈(`\n`)을 제외한 모든 문자 1개
  * `\d` : 숫자 (`[0-9]`)
  * `\w` : 문자, 숫자, 언더바 (`[a-zA-Z0-9_]`)
  * `\s` : 공백 문자 (스페이스, 탭, 줄바꿈 등)

* **반복 기호 (얼마나 반복되는지)**
  * `*` : 0번 이상 반복
  * `+` : 1번 이상 반복
  * `?` : 0번 또는 1번 등장
  * `{n}` : 정확히 $n$번 반복

* **시작과 끝 앵커 및 조건**
  * `^` : 문자열의 시작
  * `$` : 문자열의 끝
  * `|` : OR (또는) 조건

----

## 2. 필수 기본 함수 4가지 (`search`, `findall`, `sub`, `split`)

In [7]:
text = "내 전화번호는 010-1234-5678이고, 사원 번호는 A1234야."

# 1. re.search(): 전체를 탐색해 첫 번째 매칭 결과 반환 (Match 객체)
match_obj = re.search(r'\d{3}-\d{4}-\d{4}', text)
print("1. search 결과:", match_obj.group() if match_obj else "없음")

# 2. re.findall(): 일치하는 모든 부분을 찾아 리스트로 반환
numbers = re.findall(r'\d+', text)
print("2. findall 숫자 추출:", numbers)

# 3. re.sub(): 패턴과 일치하는 부분을 다른 문자열로 치환
masked_text = re.sub(r'\d{3}-\d{4}-\d{4}', '010-****-****', text)
print("3. sub 마스킹 결과:", masked_text)

# 4. re.split(): 여러 개의 구분자로 문자열 분리
split_text = "apple,banana;orange/grape"
print("4. split 결과:", re.split(r'[,;/]', split_text))

text = "apple_and_banana_or_orange"

# 소괄호를 사용하면 구분자(and, or)도 결과에 같이 포함됩니다.
result = re.split(r'(and|or)', text)
print(result)

result = re.split(r'(?:and|or)', text)
print(result)
result = re.split(r'[abc]', text)
print(result)

1. search 결과: 010-1234-5678
2. findall 숫자 추출: ['010', '1234', '5678', '1234']
3. sub 마스킹 결과: 내 전화번호는 010-****-****이고, 사원 번호는 A1234야.
4. split 결과: ['apple', 'banana', 'orange', 'grape']
['apple_', 'and', '_banana_', 'or', '_', 'or', 'ange']
['apple_', '_banana_', '_', 'ange']
['', 'pple_', 'nd_', '', 'n', 'n', '_or_or', 'nge']


----

## 3. 심화 탐색 및 Match 객체 활용 (`match`, `fullmatch`, `finditer`)

In [3]:
sample = "Python 3.10 is awesome!"

# re.match(): 문자열의 '시작 지점'부터 일치하는지 확인
print("match:", bool(re.match(r'Python', sample)))

# re.fullmatch(): 문자열 전체가 패턴과 완벽히 일치해야 함
print("fullmatch:", bool(re.fullmatch(r'Python 3\.10 is awesome!', sample)))

# re.finditer(): Match 객체 이탈레이터 반환 (위치 정보 span 활용)
for m in re.finditer(r'\d+\.\d+', sample):
    print(f"찾은 값: {m.group()}, 위치 범위: {m.span()}")

match: True
fullmatch: True
찾은 값: 3.10, 위치 범위: (7, 11)


----

## 4. 그룹(Group) 및 명명된 캡처 그룹

In [4]:
# 명명된 캡처 그룹 문법: (?P<이름>패턴)
contact_text = "연락처: 02-123-4567, 휴대폰: 010-9876-5432"
pattern = re.compile(r'(?P<prefix>\d{2,3})-(?P<middle>\d{3,4})-(?P<last>\d{4})')

for m in pattern.finditer(contact_text):
    print(f"국번: {m.group('prefix')}, 뒷자리: {m.group('last')}")

국번: 02, 뒷자리: 4567
국번: 010, 뒷자리: 5432


----

## 5. 주요 플래그(Flags) 옵션
* `re.IGNORECASE` (또는 `re.I`) : 대소문자 구분을 무시
* `re.MULTILINE` (또는 `re.M`) : 여러 줄에 걸쳐 `^`, `$` 앵커 적용

In [5]:
multiline_text = "First line\nSecond line\nTHIRD LINE"

# 대소문자 무시 검색
found_ignore = re.findall(r'line', multiline_text, re.IGNORECASE)
print("대소문자 무시 'line' 개수:", len(found_ignore))

대소문자 무시 'line' 개수: 3


----

## 6. 실전 응용: 프로그래머스 '영어가 싫어요' 정규식 풀이

정규표현식의 `re.sub`와 람다 함수를 조합하여 영단어를 한 번에 숫자로 치환하는 우아한 풀이입니다.

In [6]:
def solution(numbers):
    # 영단어와 대응하는 숫자를 담은 딕셔너리
    digit = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9'
    }
    
    # 딕셔너리의 키들을 '|'(OR) 기호로 묶어서 하나의 패턴으로 생성
    pattern = '|'.join(digit.keys())
    
    # re.sub를 이용해 패턴과 일치하는 부분을 찾아 딕셔너리의 값으로 치환
    result = re.sub(pattern, lambda m: digit[m.group(0)], numbers)
    
    return int(result)

# 테스트 실행
test_input = "onetwothreefourfivesixseveneightnine"
print(f"입력: {test_input}")
print(f"결과: {solution(test_input)}")

입력: onetwothreefourfivesixseveneightnine
결과: 123456789
